# DWBI Week 3 — Fact & Dimension Tables
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's flat `Shipments_Unified` (34 rows, 578,650 kg) becomes **one fact table + five dimensions** — and the totals do not move by a single kilogram.

## T5 — The other four dimensions

### Block A — DimRoute and the trap

The brief says "Remove Duplicates → 20 rows". On the **raw** Origin→Destination strings you get **30** —
because `Karachi Port → Lahore Dry Port` and `Port Qasim → Lahore Dry Port` are different strings for the
*same conformed route* (Karachi → Lahore). **Conform to CityKeys first, dedupe second → 20 rows.**
Minting a surrogate key for a raw string that isn't a real thing creates a phantom row in every report.

`DimRoute`: 20 rows + Unknown — `RouteKey, OriginCityKey, DestCityKey, RouteName ("Karachi -> Lahore"), Corridor, DistanceBand`.

### Block B — DimMode & DimStatus (typed by hand — reference data)

DimMode: Marine / Rail / Road (+ Unknown) with Carrier, TonnageClass, CostProfile, IsContainerised.
DimStatus: Delivered / In Transit / Delayed (+ Unknown) with IsClosed, IsException, **DisplayOrder**
(`StatusName` sorts by it, so every visual orders itself).

### Block C — DimDate: 28 rows, not 12

All 28 days of February 2026 — including the 16 quiet days. A date dimension is a *calendar*, not an
index of the fact table: the quiet days are exactly what lets a report show "zero shipments on 8 Feb"
instead of pretending the day never existed.

**Q1** — two city keys ≠ snowflake: two role-playing *references* into the same DimCity, not normalised sub-tables.
**Q2** — Mode stays its own dimension: it describes *how*, not *where* — Karachi→Lahore by sea and rail is one
route with different economics, and every future fact reuses the conformed Mode.
**Q3** — Karachi→Lahore via Sukkur: **new row** — DimRoute's grain is the travelled path, not the city pair.
**Q4** — answered above: calendars have no holes.

In [1]:
import pandas as pd
r = pd.read_csv("data/DimRoute.csv"); d = pd.read_csv("data/DimDate.csv")
print(f"DimRoute: {len(r)} rows | DimDate: {len(d)} rows")
print("Distance bands:", r[r.RouteKey!=-1].DistanceBand.value_counts().to_dict())
print("Sample:", r.loc[r.RouteKey==1, ["RouteName","Corridor","DistanceBand"]].to_dict("records")[0])

DimRoute: 21 rows | DimDate: 28 rows
Distance bands: {'Long (>800 km)': 7, 'Short (<300 km)': 7, 'Medium (300-800 km)': 6}
Sample: {'RouteName': 'Karachi -> Lahore', 'Corridor': 'South-Central', 'DistanceBand': 'Long (>800 km)'}
